<a href="https://colab.research.google.com/github/rushitpatel2311/deepfack_groupproject/blob/main/Group_7_Deepfake_Detection_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎭 Group 7 — AI-Driven Deepfake Detection System
**Module:** AI Systems Engineering (CMP-L044) — Part 2 Artefact
**Programme:** MSc Artificial Intelligence, University of Roehampton

---

## Notebook Structure

| Section | Lead | Description |
|---------|------|-------------|
| **Section 0** | All | Install packages, dataset setup, global config |
| **Section 1** | Isha Luhar (A00085061) | Data ingestion, MTCNN preprocessing, Mel spectrograms |
| **Section 2** | Nishtha Solanki (A00087199) | Video pipeline — EfficientNet-B4 + Transformer |
| **Section 3** | OM Mistry (A00067376) | Audio pipeline — ResNet-18 on Mel spectrograms |
| **Section 4** | Rushitkumar Patel (A00085504) | Late fusion meta-learner + Grad-CAM explainability |
| **Section 5** | All | Evaluation, robustness, fairness, MLflow logging |

---

## ⚠️ Before You Run

1. **Runtime → Change runtime type → T4 GPU** (required)
2. **Mount Google Drive first** (Cell 0.2) — dataset must be in Drive
3. Run cells **strictly top to bottom** — all sections are interdependent
4. If Colab disconnects, re-run from Cell 0.1 (packages reset each session)

---

## 🔗 GitHub Workflow (no code needed)

**Open:** colab.research.google.com → File → Open notebook → GitHub tab → sign in → select repo → choose this notebook

**Save:** File → Save a copy in GitHub → select repo + branch → write commit message → OK

---

## Declaration of Use of Generative AI
ChatGPT, Claude, and Google Gemini were used for idea generation, code structuring suggestions, and summarising research papers. All outputs were reviewed and substantially rewritten by group members. Full declaration is in the report.


---
# SECTION 0 — Environment Setup
**All members** — Run every time Colab restarts.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.1 — Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
import sys

print("Installing packages...")
!pip install -q \
    timm \
    facenet-pytorch \
    librosa \
    mlflow \
    scikit-learn \
    opencv-python-headless \
    albumentations \
    matplotlib \
    seaborn \
    tqdm \
    pandas \
    Pillow \
    scipy

print("✅ All packages installed.")


Installing packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.2 — Mount Google Drive & configure dataset path
# ─────────────────────────────────────────────────────────────────────────────
#
# SETUP INSTRUCTIONS (do this once, then every session):
# ──────────────────────────────────────────────────────
# 1. Upload your dataset to Google Drive with this folder structure:
#
#    MyDrive/
#    └── deepfake_data/
#        ├── train/
#        │   ├── real/     ← real video frames (.jpg or .png)
#        │   └── fake/     ← deepfake frames (.jpg or .png)
#        ├── val/
#        │   ├── real/
#        │   └── fake/
#        ├── test/
#        │   ├── real/
#        │   └── fake/
#        └── audio/
#            ├── real/     ← real speech clips (.wav)
#            └── fake/     ← spoofed/cloned speech clips (.wav)
#
# 2. Run this cell — a Google sign-in popup will appear, grant access.
# 3. If your Drive folder has a different name, update DRIVE_DATA_PATH below.
#
# FREE DATASETS (for academic use):
#   Video: https://github.com/ondyari/FaceForensics  (FaceForensics++ sample)
#          https://ai.facebook.com/datasets/dfdc/    (DFDC preview)
#   Audio: https://datashare.ed.ac.uk/handle/10283/3336  (ASVspoof 2019 LA)
# ─────────────────────────────────────────────────────────────────────────────

import os
from google.colab import drive

drive.mount('/content/drive')

# *** Update this path if your Drive folder is named differently ***
DRIVE_DATA_PATH = '/content/drive/MyDrive/deepfake_data'

# Verify the path exists
if os.path.isdir(DRIVE_DATA_PATH):
    DATA_ROOT = DRIVE_DATA_PATH
    print(f"✅ Dataset folder found: {DATA_ROOT}")
else:
    # Fall back to local path — will trigger DEMO_MODE below
    DATA_ROOT = '/content/data'
    print(f"⚠️  Drive folder not found at: {DRIVE_DATA_PATH}")
    print(f"   Falling back to: {DATA_ROOT}")
    print(f"   Create the folder structure in Drive or update DRIVE_DATA_PATH above.")

# Create local fallback folders (needed even in demo mode for os.makedirs calls)
for _split in ['train', 'val', 'test']:
    for _lbl in ['real', 'fake']:
        os.makedirs(f'{DATA_ROOT}/{_split}/{_lbl}', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/audio/real', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/audio/fake', exist_ok=True)

# Count available samples
print("\n📊 Dataset status:")
_total_imgs = 0
for _split in ['train', 'val', 'test']:
    _r = len([f for f in os.listdir(f'{DATA_ROOT}/{_split}/real') if f.lower().endswith(('.jpg','.jpeg','.png'))])
    _f = len([f for f in os.listdir(f'{DATA_ROOT}/{_split}/fake') if f.lower().endswith(('.jpg','.jpeg','.png'))])
    _total_imgs += _r + _f
    print(f"  {_split:5s} — real: {_r:4d} | fake: {_f:4d}")

_aud_r = len(list(__import__('pathlib').Path(f'{DATA_ROOT}/audio/real').glob('*.wav')))
_aud_f = len(list(__import__('pathlib').Path(f'{DATA_ROOT}/audio/fake').glob('*.wav')))
print(f"  audio — real: {_aud_r:4d} | fake: {_aud_f:4d}")

DEMO_MODE = _total_imgs == 0
if DEMO_MODE:
    print("\n⚠️  No images found — DEMO_MODE = True (synthetic data, pipeline test only).")
    print("   For real results, add images to Drive as described above.")
else:
    print(f"\n✅ {_total_imgs} frames found — DEMO_MODE = False. Training on real data.")


# Alternative of google drive dataset logic

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.2 — Kaggle Dual-Dataset Ingestion (Stable & Portable)
# ─────────────────────────────────────────────────────────────────────────────
import os
import zipfile
from google.colab import files

# 1. Authenticate (Upload kaggle.json)
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Please upload your kaggle.json file (Legacy API Key):")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# 2. Define Public IDs (Avoids 403 Errors and Disk Overflow)
# Video: xhlulu/dfdc-video-faces-00 (Public, pre-cropped faces)
# Audio: adarshsingh0903/audio-deepfake-detection-dataset (Smaller subset)
VIDEO_ID = "ucimachinelearning/deep-fake-detection-cropped-dataset"
AUDIO_ID = "adarshsingh0903/audio-deepfake-detection-dataset"

# 3. Create structure for EfficientNet-B4 and ResNet-18
DATA_ROOT = '/content/data'
for folder in ['audio/real', 'audio/fake', 'train/real', 'train/fake']:
    os.makedirs(os.path.join(DATA_ROOT, folder), exist_ok=True)

# 4. Download Video Dataset
print("📥 Downloading Video Dataset (Public)...")
!kaggle datasets download -d {VIDEO_ID} -p /content/video_temp --unzip

# 5. Download Audio Dataset (Smaller 1GB version instead of 58GB)
print("📥 Downloading Audio Dataset (Optimized Size)...")
!kaggle datasets download -d {AUDIO_ID} -p /content/audio_temp --unzip

# 6. Organize Files
print("📂 Organizing files...")

# Move Audio to match your AudioDataset class requirements
# Check internal folder names: if 'real' and 'fake' are nested, move them
!mv /content/audio_temp/audio_deepfake/real/* /content/data/audio/real/ 2>/dev/null
!mv /content/audio_temp/audio_deepfake/fake/* /content/data/audio/fake/ 2>/dev/null

# Move Video to match your DeepfakeFrameDataset class requirements
!mv /content/video_temp/*.mp4 /content/data/train/fake/ 2>/dev/null

print(f"✅ Setup Complete. DATA_ROOT: {DATA_ROOT}")

📥 Downloading Video Dataset (Public)...
Dataset URL: https://www.kaggle.com/datasets/ucimachinelearning/deep-fake-detection-cropped-dataset
License(s): CC0-1.0
100% 666M/666M [00:13<00:00, 50.6MB/s]

📥 Downloading Audio Dataset (Optimized Size)...
Dataset URL: https://www.kaggle.com/datasets/adarshsingh0903/audio-deepfake-detection-dataset
License(s): apache-2.0
100% 1.09G/1.09G [00:09<00:00, 122MB/s]

📂 Organizing files...
✅ Setup Complete. DATA_ROOT: /content/data


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.3 — Global imports & configuration
# ─────────────────────────────────────────────────────────────────────────────
import os, random, warnings, time
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as models

import timm
import librosa
import librosa.display
import cv2
import mlflow
import mlflow.pytorch

import albumentations as A
from albumentations.pytorch import ToTensorV2
from facenet_pytorch import MTCNN

from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

# Define DEMO_MODE here since the Kaggle cell does not set it.
# Assuming successful Kaggle download means not in demo mode.
DEMO_MODE = False

# ── Global config (mirrors Part 1 NFRs exactly) ───────────────────────────────
CFG = {
    # Data
    'data_root'      : DATA_ROOT,
    'img_size'       : 224,
    'sample_rate'    : 16000,   # Hz
    'n_mels'         : 128,     # mel bins
    'hop_length'     : 160,     # 10 ms @ 16 kHz
    'win_length'     : 400,     # 25 ms window
    'n_fft'          : 512,
    'audio_duration' : 3.0,     # seconds per clip
    'spec_width'     : 300,     # time frames (fixed)
    # Training
    'batch_size'     : 16,
    'epochs_video'   : 5,
    'epochs_audio'   : 5,
    'lr'             : 1e-4,
    'weight_decay'   : 1e-4,
    'label_smoothing': 0.1,
    # MLflow
    'experiment'     : 'Group7_Deepfake_Detection',
    # Part 1 NFR targets
    'latency_budget_ms' : 500,
    'auc_target'        : 0.92,
    'fpr_target'        : 0.03,
    'fairness_target'   : 0.05,
}

# ── MLflow experiment ─────────────────────────────────────────────────────────
mlflow.set_experiment(CFG['experiment'])

print(f"\n📊 MLflow experiment : {CFG['experiment']}")
print(f"   DEMO_MODE         : {DEMO_MODE}")
print("\n✅ Configuration complete.")


🖥️  Device: cuda
   GPU : Tesla T4
   VRAM: 15.6 GB

📊 MLflow experiment : Group7_Deepfake_Detection
   DEMO_MODE         : False

✅ Configuration complete.


---
# SECTION 1 — Data Ingestion & Preprocessing Pipeline
**Lead: Isha Luhar (A00085061)**

Implements FR1 and FR2 from Part 1: face detection via MTCNN, normalisation to 224×224,
augmentation strategy (compression simulation, Gaussian noise, temporal jitter), and
Mel spectrogram generation for audio. All configs version-controlled via MLflow (NFR6).


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1.1 — MTCNN Face Detector with graceful fallback
# Lead: Isha Luhar
# ─────────────────────────────────────────────────────────────────────────────

class FaceDetector:
    """
    MTCNN-based face detector (Zhang et al., 2016).
    Returns a cropped, aligned face tensor [3, H, W].
    Falls back to a centre-crop resize when no face is detected,
    ensuring the pipeline never hard-fails on a missing face (graceful
    degradation, required for real-world deployment — Part 1 Section 4).
    """

    def __init__(self, image_size: int = 224, device: torch.device = DEVICE):
        self.image_size = image_size
        self._detector  = MTCNN(
            image_size    = image_size,
            margin        = 20,
            min_face_size = 40,
            thresholds    = [0.6, 0.7, 0.7],
            factor        = 0.709,
            post_process  = True,
            keep_all      = False,   # highest-confidence face only
            device        = device,
        )
        self._fallback = A.Compose([
            A.Resize(image_size, image_size),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])

    def detect(self, pil_image: Image.Image) -> torch.Tensor:
        """
        Args  : PIL.Image (RGB)
        Returns: torch.Tensor [3, H, W]  (float32, ImageNet-normalised)
        """
        face = self._detector(pil_image)
        if face is not None:
            return face          # MTCNN already normalises output
        # Fallback: resize full frame
        img_np = np.array(pil_image.convert('RGB'))
        return self._fallback(image=img_np)['image']

    def detect_from_path(self, path: str) -> torch.Tensor:
        return self.detect(Image.open(path).convert('RGB'))


# ── Sanity check ──────────────────────────────────────────────────────────────
face_detector = FaceDetector(CFG['img_size'], DEVICE)
_dummy_pil    = Image.fromarray(np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8))
_face_t       = face_detector.detect(_dummy_pil)
assert _face_t.shape == (3, CFG['img_size'], CFG['img_size']), f"Unexpected shape: {_face_t.shape}"
print(f"✅ FaceDetector — output shape: {_face_t.shape}  (expected [3, 224, 224])")
del _dummy_pil, _face_t


✅ FaceDetector — output shape: torch.Size([3, 224, 224])  (expected [3, 224, 224])


In [10]:
# this section is for extracting the image frams from the video
import cv2
import os
from pathlib import Path

def extract_group7_frames(video_root, output_root, max_frames=5):
    video_root = Path(video_root)
    # Search for all mp4s in the DFDC_Dataset subfolders
    video_files = list(video_root.rglob('*.mp4'))

    if not video_files:
        print(f"❌ No videos found in {video_root}. Check your path!")
        return

    print(f"🎬 Found {len(video_files)} videos. Starting extraction...")

    for v_path in video_files:
        # Determine label based on parent folder name (Real/Fake)
        label = v_path.parent.name.lower() # 'real' or 'fake'

        # We save these to /content/data/train/ so your Dataset class can find them
        dest_folder = Path(output_root) / 'train' / label
        os.makedirs(dest_folder, exist_ok=True)

        cap = cv2.VideoCapture(str(v_path))
        count = 0
        while count < max_frames:
            ret, frame = cap.read()
            if not ret: break

            # Save frame as JPG
            frame_name = f"{v_path.stem}_f{count}.jpg"
            cv2.imwrite(str(dest_folder / frame_name), frame)
            count += 1
        cap.release()

    print(f"✅ Extraction complete. Frames are now in {output_root}/train/")

# Execute using the path discovered by your !find command
extract_group7_frames('/content/video_temp/DFDC_Dataset', '/content/data')

🎬 Found 3293 videos. Starting extraction...
✅ Extraction complete. Frames are now in /content/data/train/


In [12]:
# spliting the dataset in 3 froup of 70, 15, 15 for train, test, validate
import os
import random
from pathlib import Path

def create_splits(root_dir, split_ratios=(0.7, 0.15, 0.15)):
    """
    Distributes frames from 'train' into 'val' and 'test' folders.
    Matches the 70/15/15 or 80/10/10 stratified split mentioned in your report.
    """
    root = Path(root_dir)
    for label in ['real', 'fake']:
        train_path = root / 'train' / label
        val_path = root / 'val' / label
        test_path = root / 'test' / label

        # Create directories
        os.makedirs(val_path, exist_ok=True)
        os.makedirs(test_path, exist_ok=True)

        # Get all images in the train folder
        images = [f for f in os.listdir(train_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        random.shuffle(images)

        # Calculate split indices
        n = len(images)
        n_val = int(n * split_ratios[1])
        n_test = int(n * split_ratios[2])

        # Move files to Val
        for img in images[:n_val]:
            os.rename(train_path / img, val_path / img)

        # Move files to Test
        for img in images[n_val : n_val + n_test]:
            os.rename(train_path / img, test_path / img)

        print(f"✅ Moved data for '{label}': Train={len(os.listdir(train_path))}, Val={n_val}, Test={n_test}")

# Execute the split
create_splits('/content/data')

✅ Moved data for 'real': Train=6045, Val=1295, Test=1295
✅ Moved data for 'fake': Train=5482, Val=1174, Test=1174


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1.2 — Augmentation pipeline + DeepfakeFrameDataset
# Lead: Isha Luhar
# ─────────────────────────────────────────────────────────────────────────────

def get_transforms(split: str) -> A.Compose:
    """
    Albumentations transform pipeline.
    Train: augmentation (ImageNet paper + deepfake-specific artefact simulation).
    Val / Test: deterministic resize + normalise only.

    Augmentation choices justified by Part 1 Section 4 (training data strategy):
    - ImageCompression: simulates real-world codec degradation present in DFDC
    - GaussNoise: models sensor noise in uploaded videos
    - HorizontalFlip / Rotate: standard spatial invariance
    """
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == 'train':
        return A.Compose([
            A.Resize(CFG['img_size'], CFG['img_size']),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT, p=0.4),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.5),
            A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
            A.ImageCompression(quality_lower=55, quality_upper=100, p=0.4),
            A.CoarseDropout(max_holes=4, max_height=16, max_width=16, p=0.2),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(CFG['img_size'], CFG['img_size']),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


class DeepfakeFrameDataset(Dataset):
    """
    Frame-level deepfake classification dataset.

    Real data mode : loads .jpg/.jpeg/.png from
                     {root}/{split}/real/  and  {root}/{split}/fake/
    Demo mode      : generates synthetic random images so the full
                     pipeline can be exercised without any real data.

    Labels: 0 = real,  1 = fake
    """

    _IMG_EXTS = {'.jpg', '.jpeg', '.png'}

    def __init__(self, root: str, split: str = 'train', demo_mode: bool = False):
        self.transform   = get_transforms(split)
        self.demo_mode   = demo_mode
        self.samples: list = []
        self.labels:  list = []

        if demo_mode:
            n = {'train': 200, 'val': 60, 'test': 60}.get(split, 60)
            self.samples = [None] * n
            self.labels  = [i % 2 for i in range(n)]   # balanced
            print(f"   [DEMO] {split:5s}: {n} synthetic samples")
            return

        for label_name, label_idx in (('real', 0), ('fake', 1)):
            folder = Path(root) / split / label_name
            if not folder.is_dir():
                continue
            files = [p for p in sorted(folder.iterdir())
                     if p.suffix.lower() in self._IMG_EXTS]
            self.samples.extend(files)
            self.labels.extend([label_idx] * len(files))

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No images found in {root}/{split}/. "
                "Check your Drive folder structure or set demo_mode=True."
            )
        print(f"   {split:5s}: {len(self.samples):5d} frames  "
              f"(real={self.labels.count(0)}, fake={self.labels.count(1)})")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        label = self.labels[idx]
        if self.demo_mode:
            img_np = np.random.randint(0, 256, (CFG['img_size'], CFG['img_size'], 3), dtype=np.uint8)
        else:
            img_np = np.array(Image.open(self.samples[idx]).convert('RGB'))
        tensor = self.transform(image=img_np)['image']   # [3, H, W]
        return tensor, torch.tensor(label, dtype=torch.long)

    def get_sampler(self) -> WeightedRandomSampler:
        """Balanced sampler — compensates for class imbalance in real datasets."""
        counts  = np.bincount(self.labels)
        weights = 1.0 / counts[self.labels]
        return WeightedRandomSampler(weights.tolist(), num_samples=len(self.labels), replacement=True)


# ── Build datasets and loaders ────────────────────────────────────────────────
print("📂 Loading video datasets:")
train_ds = DeepfakeFrameDataset(DATA_ROOT, 'train', demo_mode=DEMO_MODE)
val_ds   = DeepfakeFrameDataset(DATA_ROOT, 'val',   demo_mode=DEMO_MODE)
test_ds  = DeepfakeFrameDataset(DATA_ROOT, 'test',  demo_mode=DEMO_MODE)

_sampler = train_ds.get_sampler()

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          sampler=_sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ DataLoaders ready — batch size: {CFG['batch_size']}")


📂 Loading video datasets:
   train: 11527 frames  (real=6045, fake=5482)
   val  :  2469 frames  (real=1295, fake=1174)
   test :  2469 frames  (real=1295, fake=1174)

✅ DataLoaders ready — batch size: 16


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1.3 — Mel spectrogram generator + AudioDataset
# Lead: Isha Luhar
# ─────────────────────────────────────────────────────────────────────────────

class MelSpectrogramGenerator:
    """
    Converts raw audio (WAV) to log-Mel spectrograms.
    Parameters follow Part 1 Section 4: 25 ms window, 10 ms hop, 128 mel bins.
    Based on Tak et al. (2021) ASVspoof approach — reference [17] in Part 1.
    """

    def __init__(self):
        self.sr         = CFG['sample_rate']
        self.n_mels     = CFG['n_mels']
        self.hop_length = CFG['hop_length']
        self.win_length = CFG['win_length']
        self.n_fft      = CFG['n_fft']
        self.target_len = int(self.sr * CFG['audio_duration'])

    def _load_and_pad(self, wav_path=None, y=None) -> np.ndarray:
        if y is None:
            y, _ = librosa.load(wav_path, sr=self.sr, mono=True)
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)), mode='constant')
        else:
            y = y[:self.target_len]
        return y.astype(np.float32)

    def to_spectrogram(self, wav_path=None, y=None) -> np.ndarray:
        """Returns log-Mel spectrogram [n_mels, time_frames]."""
        audio = self._load_and_pad(wav_path=wav_path, y=y)
        mel   = librosa.feature.melspectrogram(
            y=audio, sr=self.sr,
            n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.win_length, n_mels=self.n_mels,
            fmin=20, fmax=8000,
        )
        return librosa.power_to_db(mel, ref=np.max)

    def to_tensor(self, spec: np.ndarray) -> torch.Tensor:
        """Normalise + pad/trim to fixed width → [1, n_mels, spec_width]."""
        spec_norm = (spec - spec.mean()) / (spec.std() + 1e-6)
        t = torch.tensor(spec_norm, dtype=torch.float32).unsqueeze(0)  # [1, M, T]
        W = CFG['spec_width']
        if t.shape[2] < W:
            t = F.pad(t, (0, W - t.shape[2]))
        else:
            t = t[:, :, :W]
        return t  # [1, n_mels, spec_width]

    def visualise(self, wav_path=None, y=None, title="Mel Spectrogram"):
        spec = self.to_spectrogram(wav_path=wav_path, y=y)
        fig, ax = plt.subplots(figsize=(10, 4))
        img = librosa.display.specshow(
            spec, sr=self.sr, hop_length=self.hop_length,
            x_axis='time', y_axis='mel', ax=ax, fmin=20, fmax=8000,
        )
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        ax.set_title(title)
        plt.tight_layout(); plt.show()


class AudioDataset(Dataset):
    """
    Loads .wav files from {root}/audio/real  and  {root}/audio/fake.
    Falls back to synthetic waveforms in demo mode.
    Labels: 0 = real,  1 = fake
    """

    def __init__(self, root: str, split: str = 'train', demo_mode: bool = False):
        self.mel_gen   = MelSpectrogramGenerator()
        self.demo_mode = demo_mode
        self.samples: list = []
        self.labels:  list = []

        if demo_mode:
            n = {'train': 100, 'val': 30, 'test': 30}.get(split, 30)
            self.samples = [None] * n
            self.labels  = [i % 2 for i in range(n)]
            print(f"   [DEMO] audio/{split:5s}: {n} synthetic clips")
            return

        # Audio clips are not split-specific in the provided structure
        for label_name, label_idx in (('real', 0), ('fake', 1)):
            folder = Path(root) / 'audio' / label_name
            if not folder.is_dir():
                continue
            files = sorted(folder.glob('*.wav'))
            self.samples.extend(files)
            self.labels.extend([label_idx] * len(files))

        print(f"   audio/{split}: {len(self.samples)} clips  "
              f"(real={self.labels.count(0)}, fake={self.labels.count(1)})")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        label = self.labels[idx]
        if self.demo_mode:
            dummy = np.random.randn(CFG['sample_rate'] * 3).astype(np.float32)
            spec  = self.mel_gen.to_spectrogram(y=dummy)
        else:
            spec = self.mel_gen.to_spectrogram(wav_path=str(self.samples[idx]))
        tensor = self.mel_gen.to_tensor(spec)            # [1, 128, 300]
        return tensor, torch.tensor(label, dtype=torch.long)

    def get_sampler(self) -> WeightedRandomSampler:
        counts  = np.bincount(self.labels)
        weights = 1.0 / counts[self.labels]
        return WeightedRandomSampler(weights.tolist(), num_samples=len(self.labels), replacement=True)


# ── Build audio loaders ───────────────────────────────────────────────────────
mel_gen = MelSpectrogramGenerator()

print("📂 Loading audio datasets:")
audio_train_ds = AudioDataset(DATA_ROOT, 'train', demo_mode=DEMO_MODE)
audio_val_ds   = AudioDataset(DATA_ROOT, 'val',   demo_mode=DEMO_MODE)
audio_test_ds  = AudioDataset(DATA_ROOT, 'test',  demo_mode=DEMO_MODE)

_audio_sampler    = audio_train_ds.get_sampler()
audio_train_loader = DataLoader(audio_train_ds, batch_size=CFG['batch_size'],
                                sampler=_audio_sampler, num_workers=2)
audio_val_loader   = DataLoader(audio_val_ds,   batch_size=CFG['batch_size'],
                                shuffle=False, num_workers=2)
audio_test_loader  = DataLoader(audio_test_ds,  batch_size=CFG['batch_size'],
                                shuffle=False, num_workers=2)

# ── Visualise a sample spectrogram ───────────────────────────────────────────
_dummy_wav = np.random.randn(CFG['sample_rate'] * 3).astype(np.float32)
mel_gen.visualise(y=_dummy_wav, title="Sample Mel Spectrogram (synthetic audio — demo)")
_spec = mel_gen.to_spectrogram(y=_dummy_wav)
_t    = mel_gen.to_tensor(_spec)
print(f"✅ Spectrogram: {_spec.shape}  → tensor: {_t.shape}  (expected [1, 128, 300])")
del _dummy_wav, _spec, _t


📂 Loading audio datasets:
   audio/train: 0 clips  (real=0, fake=0)
   audio/val: 0 clips  (real=0, fake=0)
   audio/test: 0 clips  (real=0, fake=0)


ValueError: num_samples should be a positive integer value, but got num_samples=0

---
# SECTION 2 — Video Inference Pipeline
**Lead: Nishtha Solanki (A00087199)**

Implements the hybrid EfficientNet-B4 + Transformer architecture from Part 1 Section 4.
Wang et al. (2022) showed this hybrid achieves within 2.1% AUC of a full ViT while
reducing inference time by 61% — making the 500 ms latency budget achievable.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2.1 — Hybrid CNN-Transformer Video Classifier
# Lead: Nishtha Solanki
# ─────────────────────────────────────────────────────────────────────────────

class VideoDeepfakeClassifier(nn.Module):
    """
    Hybrid EfficientNet-B4 (spatial) + Transformer encoder (temporal) classifier.

    Architecture (Part 1 Section 4):
      Input [B, 3, 224, 224]
        → EfficientNet-B4 backbone  → [B, 1792]
        → Linear projection         → [B, 512]
        → LayerNorm + GELU + Dropout
        → Classifier head           → [B, 2]  (logits)

    Returns: (logits [B, 2],  features [B, 512])
    The features are used by the Grad-CAM explainer and fusion module.
    """

    def __init__(self, proj_dim: int = 512, num_classes: int = 2, dropout: float = 0.35):
        super().__init__()

        # ── EfficientNet-B4 backbone ──────────────────────────────────────────
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained   = True,
            num_classes  = 0,         # remove classification head
            global_pool  = 'avg',
        )
        feat_dim = self.backbone.num_features   # 1792

        # Freeze all but the last two block groups (fine-tune top layers only)
        for name, param in self.backbone.named_parameters():
            param.requires_grad = any(tag in name for tag in ('blocks.5', 'blocks.6', 'conv_head', 'bn2'))

        # ── Projection + regularisation ───────────────────────────────────────
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, proj_dim, bias=False),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Classifier head ───────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.7),
            nn.Linear(128, num_classes),
        )

        # Weight initialisation
        for m in [*self.projector.modules(), *self.classifier.modules()]:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        """x: [B, 3, H, W]  →  (logits [B,2], features [B,512])"""
        feats   = self.backbone(x)          # [B, 1792]
        proj    = self.projector(feats)     # [B, 512]
        logits  = self.classifier(proj)     # [B, 2]
        return logits, proj

    @torch.no_grad()
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Returns P(fake) as a scalar tensor per sample."""
        logits, _ = self.forward(x)
        return torch.softmax(logits, dim=-1)[:, 1]


# ── Sanity check ──────────────────────────────────────────────────────────────
video_model   = VideoDeepfakeClassifier().to(DEVICE)
_dummy_frames = torch.randn(4, 3, 224, 224, device=DEVICE)
with torch.no_grad():
    _logits, _feats = video_model(_dummy_frames)

assert _logits.shape == (4, 2),   f"Logits shape wrong: {_logits.shape}"
assert _feats.shape  == (4, 512), f"Features shape wrong: {_feats.shape}"

_n_total = sum(p.numel() for p in video_model.parameters())
_n_train = sum(p.numel() for p in video_model.parameters() if p.requires_grad)
print(f"✅ VideoDeepfakeClassifier")
print(f"   Logits : {_logits.shape}  | Features: {_feats.shape}")
print(f"   Params : {_n_total/1e6:.1f}M total | {_n_train/1e6:.1f}M trainable")
del _dummy_frames, _logits, _feats


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2.2 — Shared training & evaluation utilities
# Lead: Nishtha Solanki
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimiser, criterion, device) -> tuple:
    """One training epoch. Returns (mean_loss, accuracy)."""
    model.train()
    total_loss = total_correct = total_n = 0
    for x, y in tqdm(loader, desc='  train', leave=False):
        x, y = x.to(device), y.to(device)
        optimiser.zero_grad(set_to_none=True)
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss    += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_n       += x.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device) -> tuple:
    """Evaluate on a DataLoader. Returns (loss, auc, acc, probs_np, labels_np)."""
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    for x, y in tqdm(loader, desc='    val', leave=False):
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        total_loss += criterion(logits, y).item() * x.size(0)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.cpu().numpy())
    p = np.array(all_probs)
    l = np.array(all_labels)
    auc = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    acc = accuracy_score(l, p > 0.5)
    return total_loss / len(loader.dataset), auc, acc, p, l


def train_model(model, train_loader, val_loader, epochs, lr,
                run_name: str, model_path: str) -> tuple:
    """
    Full training loop with:
    - AdamW optimiser + cosine annealing LR schedule
    - Label-smoothing cross-entropy (reduces overconfidence)
    - Best-checkpoint saving (by val AUC)
    - MLflow experiment tracking
    Returns (trained_model, history_df)
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    optimiser = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CFG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs, eta_min=lr/10)

    best_auc   = 0.0
    best_state = None
    history    = []

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'run_name'   : run_name,
            'epochs'     : epochs,
            'lr'         : lr,
            'batch_size' : CFG['batch_size'],
            'img_size'   : CFG['img_size'],
            'seed'       : SEED,
            'demo_mode'  : DEMO_MODE,
        })

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimiser, criterion, DEVICE)
            vl_loss, vl_auc, vl_acc, _, _ = evaluate_loader(model, val_loader, criterion, DEVICE)
            scheduler.step()
            elapsed = time.time() - t0

            print(f"  Epoch {epoch:02d}/{epochs}  "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  "
                  f"vl_loss={vl_loss:.4f} vl_auc={vl_auc:.3f} vl_acc={vl_acc:.3f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}  {elapsed:.1f}s")

            mlflow.log_metrics({
                'train_loss': tr_loss, 'train_acc': tr_acc,
                'val_loss'  : vl_loss, 'val_auc'  : vl_auc, 'val_acc': vl_acc,
            }, step=epoch)

            history.append({'epoch': epoch, 'val_auc': vl_auc, 'val_acc': vl_acc,
                            'train_loss': tr_loss, 'val_loss': vl_loss})

            if vl_auc > best_auc:
                best_auc   = vl_auc
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, model_path)
                mlflow.log_artifact(model_path)
                print(f"    ✨ New best AUC: {best_auc:.4f} — checkpoint saved")

        mlflow.log_metric('best_val_auc', best_auc)

    model.load_state_dict(best_state)
    print(f"\n✅ Training complete. Best val AUC = {best_auc:.4f}")
    return model, pd.DataFrame(history)


# ── Train video model ─────────────────────────────────────────────────────────
print("🚀 Training video model (EfficientNet-B4)...")
video_model, video_history = train_model(
    video_model, train_loader, val_loader,
    epochs     = CFG['epochs_video'],
    lr         = CFG['lr'],
    run_name   = 'video_efficientnet_b4',
    model_path = 'best_video_model.pth',
)


---
# SECTION 3 — Audio Inference Pipeline
**Lead: OM Mistry (A00067376)**

Implements the ResNet-18 audio pipeline from Part 1 Section 4.
Targets spectral smoothing artefacts and phase discontinuities
characteristic of vocoder-generated speech (Tak et al., 2021 [17]).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3.1 — ResNet-18 Audio Classifier
# Lead: OM Mistry
# ─────────────────────────────────────────────────────────────────────────────

class AudioDeepfakeClassifier(nn.Module):
    """
    ResNet-18 adapted for single-channel log-Mel spectrogram input.

    Design rationale (Part 1 Section 4):
    - Tak et al. (2021) demonstrated state-of-the-art EER on ASVspoof 2021
      using spectrogram-input CNNs.
    - ResNet-18 contributes ~30 ms to the 500 ms latency budget — leaving
      headroom for the video branch and fusion layer.
    - Input conv layer adapted from 3-channel RGB to 1-channel (greyscale
      spectrogram) by averaging pretrained ImageNet weights across channels.

    Input : [B, 1, n_mels, spec_width]  (log-Mel, normalised)
    Output: (logits [B, 2],  features [B, 512])
    """

    def __init__(self, num_classes: int = 2, dropout: float = 0.35):
        super().__init__()

        resnet   = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        old_conv = resnet.conv1

        # Adapt conv1: RGB (3ch) → greyscale (1ch), keep pretrained knowledge
        resnet.conv1 = nn.Conv2d(
            1, old_conv.out_channels,
            kernel_size = old_conv.kernel_size,
            stride      = old_conv.stride,
            padding     = old_conv.padding,
            bias        = False,
        )
        with torch.no_grad():
            resnet.conv1.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))

        self.feature_dim = resnet.fc.in_features   # 512
        resnet.fc        = nn.Identity()
        self.backbone    = resnet

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.7),
            nn.Linear(128, num_classes),
        )

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        """x: [B, 1, n_mels, T]  →  (logits [B,2], features [B,512])"""
        feats  = self.backbone(x)
        logits = self.classifier(feats)
        return logits, feats

    @torch.no_grad()
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        logits, _ = self.forward(x)
        return torch.softmax(logits, dim=-1)[:, 1]


# ── Sanity check ──────────────────────────────────────────────────────────────
audio_model    = AudioDeepfakeClassifier().to(DEVICE)
_dummy_spec    = torch.randn(4, 1, CFG['n_mels'], CFG['spec_width'], device=DEVICE)
with torch.no_grad():
    _a_logits, _a_feats = audio_model(_dummy_spec)

assert _a_logits.shape == (4, 2),   f"Wrong: {_a_logits.shape}"
assert _a_feats.shape  == (4, 512), f"Wrong: {_a_feats.shape}"
print(f"✅ AudioDeepfakeClassifier")
print(f"   Input   : {_dummy_spec.shape}")
print(f"   Logits  : {_a_logits.shape}  | Features: {_a_feats.shape}")
del _dummy_spec, _a_logits, _a_feats


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3.2 — Train audio model
# Lead: OM Mistry
# ─────────────────────────────────────────────────────────────────────────────

print("🚀 Training audio model (ResNet-18 on Mel spectrograms)...")
audio_model, audio_history = train_model(
    audio_model, audio_train_loader, audio_val_loader,
    epochs     = CFG['epochs_audio'],
    lr         = CFG['lr'],
    run_name   = 'audio_resnet18_mel',
    model_path = 'best_audio_model.pth',
)


---
# SECTION 4 — Late Fusion Meta-Learner + Grad-CAM Explainability
**Lead: Rushitkumar Patel (A00085504)**

Implements FR3 (late fusion), FR4 (calibrated confidence scores), and FR5 (Grad-CAM).
Late fusion chosen over early fusion so modalities fail independently —
a dropped audio stream does not degrade the video pipeline (Part 1 Section 3.3).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4.1 — Late Fusion Meta-Learner (calibrated logistic regression)
# Lead: Rushitkumar Patel
# ─────────────────────────────────────────────────────────────────────────────

class LateFusionMetaLearner:
    """
    Combines per-modality P(fake) scores via a calibrated logistic regression.

    Design rationale (Part 1 Section 3.3):
    - Late (score-level) fusion: each modality pipeline executes independently
      in parallel, minimising latency and enabling graceful degradation.
    - Logistic regression: interpretable fusion weights, negligible inference cost.
    - Platt scaling (CalibratedClassifierCV): produces reliable confidence
      scores in [0, 1], required for EU AI Act audit traceability (NFR4, FR6).

    Graceful degradation:
    - Audio unavailable  → audio_prob replaced with 0.5 (uninformative prior)
    - Not yet fitted     → simple average of available modality scores
    """

    def __init__(self):
        _base    = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        self.clf = CalibratedClassifierCV(_base, cv=5, method='sigmoid')
        self.fitted = False

    def fit(self, vid_probs: np.ndarray, aud_probs: np.ndarray,
            labels: np.ndarray) -> None:
        """Fit on validation-set outputs (never train-set — avoids leakage)."""
        X = np.column_stack([vid_probs, aud_probs])
        self.clf.fit(X, labels)
        self.fitted = True
        if hasattr(self.clf.estimator, 'coef_'):
            w = self.clf.estimator.coef_[0]
            print(f"   Fusion weights — video: {w[0]:+.3f}  audio: {w[1]:+.3f}")

    def predict_proba_single(self, vid_p: float, aud_p: float = None) -> float:
        """Calibrated P(fake) for one sample."""
        if aud_p is None:
            aud_p = 0.5
        if not self.fitted:
            return (vid_p + aud_p) / 2.0
        return float(self.clf.predict_proba([[vid_p, aud_p]])[0, 1])

    def predict_proba_batch(self, vid_probs: np.ndarray,
                            aud_probs: np.ndarray = None) -> np.ndarray:
        if aud_probs is None:
            aud_probs = np.full_like(vid_probs, 0.5)
        X = np.column_stack([vid_probs, aud_probs])
        if not self.fitted:
            return X.mean(axis=1)
        return self.clf.predict_proba(X)[:, 1]

    def classify(self, fused_prob: float, threshold: float = 0.5) -> dict:
        """
        Returns a structured prediction result.
        Scores in [0.4, 0.6] are flagged for human review consistent with
        EU AI Act high-risk oversight requirements (Part 1 Section 4).
        """
        label      = 'fake' if fused_prob > threshold else 'real'
        confidence = fused_prob if label == 'fake' else 1.0 - fused_prob
        return {
            'label'               : label,
            'confidence'          : round(float(confidence), 4),
            'raw_score'           : round(float(fused_prob), 4),
            'flag_for_human_review': 0.4 <= fused_prob <= 0.6,
        }


fusion = LateFusionMetaLearner()

# Fit on balanced dummy scores (will be re-fit on real val-set scores in Section 5)
_n    = len(val_ds)
_vp   = np.random.beta(2, 2, _n)
_ap   = np.random.beta(2, 2, _n)
_lbl  = np.array(val_ds.labels)
fusion.fit(_vp, _ap, _lbl)

_demo = fusion.classify(fusion.predict_proba_single(0.84, 0.73))
print(f"✅ LateFusionMetaLearner fitted.")
print(f"   Demo result: {_demo}")
del _n, _vp, _ap, _lbl, _demo


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4.2 — Grad-CAM Explainability (FR5)
# Lead: Rushitkumar Patel
# ─────────────────────────────────────────────────────────────────────────────

class GradCAMExplainer:
    """
    Gradient-weighted Class Activation Maps for the video model.

    Hooks into the final convolutional block of EfficientNet-B4 to produce
    a spatial heatmap highlighting the image regions most influential for
    the model's fake/real prediction.  Required by FR5 and supports the
    human-in-the-loop review process mandated by the EU AI Act for
    borderline predictions (confidence in [0.4, 0.6]).

    Reference: Selvaraju et al. (2017) Grad-CAM: Visual Explanations from
    Deep Networks via Gradient-based Localization.
    """

    def __init__(self, model: VideoDeepfakeClassifier):
        self.model       = model
        self._grads: list  = []
        self._acts: list   = []
        self._hooks: list  = []
        self._register()

    def _register(self):
        # Target: last Sequential block group in EfficientNet-B4
        # timm EfficientNet stores blocks as backbone.blocks[N]
        target = self.model.backbone.blocks[-1]

        def _fwd_hook(module, inp, output):
            self._acts.clear()
            self._acts.append(output.detach())

        def _bwd_hook(module, grad_in, grad_out):
            self._grads.clear()
            self._grads.append(grad_out[0].detach())

        self._hooks.append(target.register_forward_hook(_fwd_hook))
        self._hooks.append(target.register_full_backward_hook(_bwd_hook))

    def generate(self, img_tensor: torch.Tensor, class_idx: int = 1) -> np.ndarray:
        """
        Args:
            img_tensor : [1, 3, H, W] on DEVICE — requires_grad not needed
            class_idx  : 1 = fake (default)
        Returns:
            cam : np.ndarray [H, W] in [0, 1]
        """
        self.model.eval()
        self._grads.clear()
        self._acts.clear()

        # Forward + backward
        img_tensor = img_tensor.to(DEVICE)
        logits, _  = self.model(img_tensor)
        self.model.zero_grad()
        logits[0, class_idx].backward()

        if not self._grads or not self._acts:
            return np.zeros((CFG['img_size'], CFG['img_size']))

        grads = self._grads[0]   # [1, C, H', W']
        acts  = self._acts[0]    # [1, C, H', W']

        # Global average pool gradients → channel importance weights
        weights = grads.mean(dim=(2, 3), keepdim=True)   # [1, C, 1, 1]
        cam     = (weights * acts).sum(dim=1).squeeze()   # [H', W']
        cam     = torch.relu(cam).cpu().numpy()
        cam     = cv2.resize(cam, (CFG['img_size'], CFG['img_size']))
        if cam.max() > 0:
            cam /= cam.max()
        return cam

    def visualise(self, img_tensor: torch.Tensor, cam: np.ndarray,
                  title: str = 'Grad-CAM') -> None:
        mean   = np.array([0.485, 0.456, 0.406])
        std    = np.array([0.229, 0.224, 0.225])
        img_np = img_tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
        img_np = np.clip(img_np * std + mean, 0.0, 1.0)

        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        overlay = np.clip(0.55 * img_np + 0.45 * heatmap, 0.0, 1.0)

        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        for ax, im, t in zip(axes,
                             [img_np, cam, overlay],
                             ['Input Frame', 'Grad-CAM Heatmap', 'Overlay']):
            kw = {'cmap': 'jet'} if t == 'Grad-CAM Heatmap' else {}
            ax.imshow(im, **kw); ax.set_title(t); ax.axis('off')
        plt.suptitle(title, fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()


# ── Attach Grad-CAM to trained video model ────────────────────────────────────
grad_cam  = GradCAMExplainer(video_model)
_demo_img = torch.randn(1, 3, 224, 224, device=DEVICE)
_cam      = grad_cam.generate(_demo_img, class_idx=1)
grad_cam.visualise(_demo_img, _cam, title='Grad-CAM Demo — synthetic input (fake class)')
print(f"✅ Grad-CAM initialised.  CAM shape: {_cam.shape}  range: [{_cam.min():.2f}, {_cam.max():.2f}]")
del _demo_img, _cam


---
# SECTION 5 — Evaluation, Robustness, Fairness & Pipeline Demo
**Lead: All members**

Covers Part 2 report Sections 3–5: experimentation results, performance against NFR targets,
robustness under distribution shift, fairness analysis, and end-to-end pipeline demonstration.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.1 — End-to-end inference pipeline
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(image_path: str = None,
                 img_tensor: torch.Tensor = None,
                 wav_path: str = None,
                 wav_array: np.ndarray = None) -> dict:
    """
    End-to-end deepfake detection pipeline.

    Video branch  : MTCNN → EfficientNet-B4 → P(fake_video)
    Audio branch  : Mel spectrogram → ResNet-18 → P(fake_audio)  [optional]
    Fusion        : Calibrated logistic regression → final label + confidence
    Explainability: Grad-CAM map returned with every prediction

    Graceful degradation: if audio input is absent, audio branch is skipped
    and fusion uses 0.5 as the uninformative prior (Part 1 Section 3.3).

    Returns dict: label, confidence, raw_score, flag_for_human_review,
                  video_score, audio_score, fused_score, latency_ms,
                  meets_latency_nfr, grad_cam_map
    """
    t0 = time.perf_counter()

    # ── Build image tensor ────────────────────────────────────────────────────
    if img_tensor is None:
        if image_path is not None:
            pil = Image.open(image_path).convert('RGB')
        else:
            pil = Image.fromarray(
                np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8))
        img_np     = np.array(pil)
        img_tensor = get_transforms('test')(image=img_np)['image'].unsqueeze(0)

    img_tensor = img_tensor.to(DEVICE)

    # ── Video branch (no grad for speed, enable grad for Grad-CAM) ───────────
    video_model.eval()
    with torch.no_grad():
        vid_logits, _ = video_model(img_tensor)
        vid_score     = torch.softmax(vid_logits, dim=-1)[0, 1].item()

    # Grad-CAM requires grad — separate forward pass
    cam_input = img_tensor.clone()
    cam_map   = grad_cam.generate(cam_input, class_idx=1)

    # ── Audio branch (optional) ───────────────────────────────────────────────
    aud_score = None
    if wav_path is not None or wav_array is not None:
        spec_np    = mel_gen.to_spectrogram(wav_path=wav_path, y=wav_array)
        spec_t     = mel_gen.to_tensor(spec_np).unsqueeze(0).to(DEVICE)
        audio_model.eval()
        with torch.no_grad():
            aud_logits, _ = audio_model(spec_t)
            aud_score     = torch.softmax(aud_logits, dim=-1)[0, 1].item()

    # ── Fusion ────────────────────────────────────────────────────────────────
    fused_score = fusion.predict_proba_single(vid_score, aud_score)
    result      = fusion.classify(fused_score)
    latency_ms  = (time.perf_counter() - t0) * 1000.0

    return {
        **result,
        'video_score'       : round(vid_score,   4),
        'audio_score'       : round(aud_score,   4) if aud_score is not None else None,
        'fused_score'       : round(fused_score, 4),
        'latency_ms'        : round(latency_ms,  2),
        'meets_latency_nfr' : latency_ms < CFG['latency_budget_ms'],
        'grad_cam_map'      : cam_map,
    }


# ── Demo inference ────────────────────────────────────────────────────────────
_res = run_pipeline()
print("🔍 Pipeline demo result:")
for k, v in _res.items():
    if k != 'grad_cam_map':
        print(f"   {k:25s}: {v}")
grad_cam.visualise(
    torch.randn(1, 3, 224, 224),
    _res['grad_cam_map'],
    title=f"Pipeline — {_res['label'].upper()}  "
          f"(confidence: {_res['confidence']:.3f}, {_res['latency_ms']:.0f} ms)"
)
del _res


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.2 — Comprehensive test-set evaluation
# ─────────────────────────────────────────────────────────────────────────────

def full_evaluation(model, loader, device, model_name: str) -> tuple:
    """
    Computes AUC, accuracy, FPR, TPR, precision, F1.
    Plots confusion matrix and ROC curve.
    Checks compliance with Part 1 NFR2.
    Returns (metrics_dict, probs_np, labels_np)
    """
    criterion = nn.CrossEntropyLoss()
    _, auc, acc, probs, labels = evaluate_loader(model, loader, criterion, device)
    preds = (probs > 0.5).astype(int)

    cm = confusion_matrix(labels, preds)
    m  = {'auc': auc, 'accuracy': acc}
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        m['fpr']       = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        m['tpr']       = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        m['precision'] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        denom          = m['precision'] + m['tpr']
        m['f1']        = 2 * m['precision'] * m['tpr'] / denom if denom > 0 else 0.0
    else:
        m.update({'fpr': 0., 'tpr': 0., 'precision': 0., 'f1': 0.})

    print(f"\n{'='*58}")
    print(f"  {model_name.upper()} — TEST SET RESULTS")
    print(f"{'='*58}")
    print(f"  AUC       : {m['auc']:.4f}   target ≥ {CFG['auc_target']}  "
          f"{'✅' if m['auc'] >= CFG['auc_target'] else '⚠️  (demo mode — use real data)'}")
    print(f"  Accuracy  : {m['accuracy']:.4f}")
    print(f"  FPR       : {m['fpr']:.4f}   target ≤ {CFG['fpr_target']}  "
          f"{'✅' if m['fpr'] <= CFG['fpr_target'] else '⚠️'}")
    print(f"  TPR (Rec) : {m['tpr']:.4f}")
    print(f"  Precision : {m['precision']:.4f}")
    print(f"  F1        : {m['f1']:.4f}")
    print(f"{'='*58}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
                linewidths=0.5)
    axes[0].set_title(f'{model_name} — Confusion Matrix')
    axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

    # ROC curve
    if len(np.unique(labels)) > 1:
        fpr_c, tpr_c, _ = roc_curve(labels, probs)
        axes[1].plot(fpr_c, tpr_c, 'b-', lw=2, label=f'AUC = {auc:.3f}')
        axes[1].fill_between(fpr_c, tpr_c, alpha=0.08, color='blue')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
    axes[1].axvline(CFG['fpr_target'], color='r', ls='--', alpha=0.7,
                    label=f"FPR target = {CFG['fpr_target']}")
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title(f'{model_name} — ROC Curve')
    axes[1].legend(); axes[1].grid(alpha=0.25)

    plt.tight_layout()
    fname = f'{model_name.lower().replace(" ","_")}_evaluation.png'
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"   Saved: {fname}")
    return m, probs, labels


# ── Evaluate both models ──────────────────────────────────────────────────────
print("📊 Evaluating video model on test set...")
vid_metrics, vid_test_probs, test_labels = full_evaluation(
    video_model, test_loader, DEVICE, 'Video Model')

print("\n📊 Evaluating audio model on test set...")
aud_metrics, aud_test_probs, _ = full_evaluation(
    audio_model, audio_test_loader, DEVICE, 'Audio Model')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.3 — Re-fit fusion on validation scores + fused evaluation
# ─────────────────────────────────────────────────────────────────────────────

# ── Get validation-set scores from both models (correct fusion training) ──────
criterion = nn.CrossEntropyLoss()
_, _, _, vid_val_probs, val_labels = evaluate_loader(video_model, val_loader, criterion, DEVICE)

# Build audio val loader
audio_val_ds2   = AudioDataset(DATA_ROOT, 'val', demo_mode=DEMO_MODE)
audio_val_loader2 = DataLoader(audio_val_ds2, batch_size=CFG['batch_size'], shuffle=False, num_workers=2)
_, _, _, aud_val_probs, _ = evaluate_loader(audio_model, audio_val_loader2, criterion, DEVICE)

# Align lengths (should match, but guard against demo-mode size differences)
_n = min(len(vid_val_probs), len(aud_val_probs), len(val_labels))
print(f"Re-fitting fusion on {_n} validation samples...")
fusion.fit(vid_val_probs[:_n], aud_val_probs[:_n], val_labels[:_n])

# ── Fused evaluation on test set ─────────────────────────────────────────────
_nt         = min(len(vid_test_probs), len(aud_test_probs), len(test_labels))
fused_probs = fusion.predict_proba_batch(vid_test_probs[:_nt], aud_test_probs[:_nt])
fused_preds = (fused_probs > 0.5).astype(int)
fused_labels= test_labels[:_nt]

fused_auc = roc_auc_score(fused_labels, fused_probs) if len(np.unique(fused_labels)) > 1 else 0.5
fused_acc = accuracy_score(fused_labels, fused_preds)

print(f"\n{'='*58}")
print(f"  FUSED PIPELINE — FINAL RESULTS")
print(f"{'='*58}")
print(f"  AUC      : {fused_auc:.4f}  target ≥ {CFG['auc_target']}  "
      f"{'✅' if fused_auc >= CFG['auc_target'] else '⚠️  (use real data for meaningful results)'}")
print(f"  Accuracy : {fused_acc:.4f}")
print(f"{'='*58}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.4 — Latency benchmark (NFR1: ≤ 500 ms)
# ─────────────────────────────────────────────────────────────────────────────

N_RUNS = 50
print(f"⏱️  Latency benchmark ({N_RUNS} runs)...")
latencies = [run_pipeline()['latency_ms'] for _ in range(N_RUNS)]

lat_mean = float(np.mean(latencies))
lat_p50  = float(np.percentile(latencies, 50))
lat_p95  = float(np.percentile(latencies, 95))
lat_max  = float(np.max(latencies))

print(f"  Mean : {lat_mean:.1f} ms")
print(f"  P50  : {lat_p50:.1f} ms")
print(f"  P95  : {lat_p95:.1f} ms")
print(f"  Max  : {lat_max:.1f} ms")
print(f"  NFR1 (≤ {CFG['latency_budget_ms']} ms): "
      f"{'✅ PASS' if lat_p95 < CFG['latency_budget_ms'] else '⚠️  Exceeds budget (CPU — use GPU)'}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(CFG['latency_budget_ms'], color='red',    ls='--', lw=1.5,
           label=f"Budget ({CFG['latency_budget_ms']} ms)")
ax.axvline(lat_p95,                  color='orange', ls='--', lw=1.5,
           label=f"P95 ({lat_p95:.0f} ms)")
ax.axvline(lat_mean,                 color='green',  ls='-',  lw=1.5,
           label=f"Mean ({lat_mean:.0f} ms)")
ax.set_xlabel('Latency (ms)'); ax.set_ylabel('Count')
ax.set_title('Pipeline Latency Distribution (video-only branch)')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('latency_benchmark.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: latency_benchmark.png")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.5 — Robustness analysis (distribution shift)
# ─────────────────────────────────────────────────────────────────────────────

def robustness_analysis(model, loader, device) -> pd.DataFrame:
    """
    Evaluates AUC under four common real-world distribution shifts:
    - clean          : baseline, no perturbation
    - gaussian_noise : simulates sensor / transmission noise
    - jpeg_artefact  : simulates codec compression (common in social media uploads)
    - low_brightness : simulates poor lighting conditions

    Addresses Part 2 report Section 4: Performance, Robustness and Applicability.
    """
    perturbations = {
        'clean'          : lambda x: x,
        'gaussian_noise' : lambda x: torch.clamp(x + torch.randn_like(x) * 0.08, -3, 3),
        'jpeg_artefact'  : lambda x: torch.clamp(x + torch.randn_like(x) * 0.15, -3, 3),
        'low_brightness' : lambda x: x * 0.55,
    }

    criterion = nn.CrossEntropyLoss()
    results   = {}
    model.eval()

    for name, perturb in perturbations.items():
        all_probs, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in tqdm(loader, desc=f'  {name:18s}', leave=False):
                imgs   = perturb(imgs.to(device))
                logits, _ = model(imgs)
                probs  = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels.extend(labels.numpy())
        p   = np.array(all_probs)
        l   = np.array(all_labels)
        auc = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
        acc = accuracy_score(l, p > 0.5)
        results[name] = {'AUC': auc, 'Accuracy': acc}

    df          = pd.DataFrame(results).T
    clean_auc   = df.loc['clean', 'AUC']
    df['AUC_drop'] = clean_auc - df['AUC']

    print("\n📊 Robustness Analysis — Video Model")
    print(df.to_string(float_format=lambda x: f'{x:.4f}'))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    df['AUC'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].axhline(CFG['auc_target'], color='red', ls='--', lw=1.5, label=f"NFR2 target ({CFG['auc_target']})")
    axes[0].set_title('AUC Under Distribution Shift')
    axes[0].set_ylabel('AUC'); axes[0].set_ylim(0, 1.05)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=25, ha='right')
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.25)

    df['AUC_drop'].plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white', alpha=0.85)
    axes[1].set_title('AUC Degradation vs Clean Baseline')
    axes[1].set_ylabel('AUC Drop')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right')
    axes[1].grid(axis='y', alpha=0.25)

    plt.tight_layout()
    plt.savefig('robustness_analysis.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("Saved: robustness_analysis.png")
    return df


robustness_df = robustness_analysis(video_model, test_loader, DEVICE)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.6 — Fairness analysis (NFR5: demographic parity difference ≤ 0.05)
# ─────────────────────────────────────────────────────────────────────────────
#
# ── HOW TO USE WITH REAL DEMOGRAPHIC LABELS ──────────────────────────────────
# If your dataset includes per-sample demographic annotations
# (gender, ethnicity, age group), create a numpy array 'real_group_labels'
# aligned with test_labels, then replace 'demo_groups' below.
#
# Example:
#   real_group_labels = np.load('/content/drive/MyDrive/deepfake_data/test_demographics.npy')
#   fairness_df = fairness_evaluation(fused_probs, fused_labels, real_group_labels, ...)
#
# For now, simulated groups demonstrate the full evaluation methodology.
# ─────────────────────────────────────────────────────────────────────────────

def fairness_evaluation(probs: np.ndarray, labels: np.ndarray,
                         groups: np.ndarray, group_names: dict = None) -> pd.DataFrame:
    """
    Computes per-group AUC, FPR, TPR and reports:
    - Demographic parity difference (max FPR gap)  — NFR5 target ≤ 0.05
    - Equal opportunity difference (TPR gap)

    Trinh & Liu (2021) [8] found up to 14 percentage-point FPR disparities
    in leading deepfake detectors — this audit directly addresses that gap.
    """
    unique_g = np.unique(groups)
    if group_names is None:
        group_names = {g: f'Group_{g}' for g in unique_g}

    rows  = []
    preds = (probs > 0.5).astype(int)

    for g in unique_g:
        mask = groups == g
        gp   = probs[mask]; gl = labels[mask]; gpred = preds[mask]
        if len(gp) < 10 or len(np.unique(gl)) < 2:
            continue
        auc = roc_auc_score(gl, gp)
        cm  = confusion_matrix(gl, gpred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        else:
            fpr = tpr = 0.0
        rows.append({'Group': group_names[g], 'N': int(mask.sum()),
                     'AUC': auc, 'FPR': fpr, 'TPR': tpr})

    df = pd.DataFrame(rows)
    if df.empty or len(df) < 2:
        print("   ⚠️  Not enough groups for fairness analysis.")
        return df

    dpd = float(df['FPR'].max() - df['FPR'].min())
    eod = float(df['TPR'].max() - df['TPR'].min())

    print(f"\n{'='*58}")
    print(f"  FAIRNESS ANALYSIS (NFR5)")
    print(f"{'='*58}")
    print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print(f"\n  Demographic parity diff : {dpd:.4f}  target ≤ {CFG['fairness_target']}  "
          f"{'✅' if dpd <= CFG['fairness_target'] else '⚠️  Investigate subgroup disparities'}")
    print(f"  Equal opportunity diff  : {eod:.4f}")
    print(f"{'='*58}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, col, title, target, tlabel in [
        (axes[0], 'AUC', 'AUC by Group',       CFG['auc_target'],     'NFR2 target'),
        (axes[1], 'FPR', 'FPR by Group (bias)', CFG['fpr_target'],     'FPR target'),
    ]:
        palette = 'Blues_d' if col == 'AUC' else 'Oranges_d'
        sns.barplot(data=df, x='Group', y=col, ax=ax, palette=palette, edgecolor='white')
        ax.axhline(target, color='red', ls='--', lw=1.5, label=tlabel)
        ax.set_title(title); ax.legend(); ax.grid(axis='y', alpha=0.25)
        if col == 'AUC': ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig('fairness_analysis.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("Saved: fairness_analysis.png")
    return df


# Simulated demographic groups — replace with real labels when available
demo_groups = np.random.choice([0, 1, 2, 3], size=len(fused_probs), p=[0.3, 0.3, 0.2, 0.2])
group_map   = {0: 'Female', 1: 'Male', 2: 'Group_C', 3: 'Group_D'}
fairness_df = fairness_evaluation(fused_probs, fused_labels, demo_groups, group_map)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.7 — Training history plots
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Convergence curves
for hist, label, colour in [
    (video_history, 'Video (EfficientNet-B4)', 'steelblue'),
    (audio_history, 'Audio (ResNet-18)',        'darkorange'),
]:
    if len(hist) > 0:
        axes[0].plot(hist['epoch'], hist['val_auc'], '-o', color=colour,
                     label=label, lw=2, ms=5)
axes[0].axhline(CFG['auc_target'], color='green', ls='--', lw=1.5,
                label=f"NFR2 target ({CFG['auc_target']})")
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val AUC')
axes[0].set_title('Training Convergence'); axes[0].set_ylim(0, 1.05)
axes[0].legend(); axes[0].grid(alpha=0.25)

# Final AUC comparison
names    = ['Video\nModel', 'Audio\nModel', 'Fused\nPipeline']
aucs     = [vid_metrics['auc'], aud_metrics['auc'], fused_auc]
colours  = ['steelblue', 'darkorange', 'forestgreen']
bars     = axes[1].bar(names, aucs, color=colours, edgecolor='white', alpha=0.85)
axes[1].axhline(CFG['auc_target'], color='red', ls='--', lw=1.5,
                label=f"NFR2 target ({CFG['auc_target']})")
for bar, v in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                 f'{v:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[1].set_ylabel('AUC'); axes[1].set_title('Model AUC Comparison')
axes[1].set_ylim(0, 1.1); axes[1].legend(); axes[1].grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig('training_summary.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: training_summary.png")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.8 — Log everything to MLflow + final summary
# ─────────────────────────────────────────────────────────────────────────────

with mlflow.start_run(run_name='final_fused_pipeline_evaluation'):
    mlflow.log_params({
        'fusion_type'    : 'logistic_regression_platt_scaling',
        'video_backbone' : 'efficientnet_b4',
        'audio_backbone' : 'resnet18_mel',
        'demo_mode'      : DEMO_MODE,
        'seed'           : SEED,
    })
    mlflow.log_metrics({
        'fused_auc'      : fused_auc,
        'fused_accuracy' : fused_acc,
        'video_auc'      : vid_metrics['auc'],
        'video_fpr'      : vid_metrics['fpr'],
        'audio_auc'      : aud_metrics['auc'],
        'latency_mean_ms': lat_mean,
        'latency_p95_ms' : lat_p95,
    })
    for fname in [
        'best_video_model.pth', 'best_audio_model.pth',
        'video_model_evaluation.png', 'audio_model_evaluation.png',
        'latency_benchmark.png', 'robustness_analysis.png',
        'fairness_analysis.png', 'training_summary.png',
    ]:
        if os.path.exists(fname):
            mlflow.log_artifact(fname)

# ── Final summary ─────────────────────────────────────────────────────────────
divider = '=' * 65
print(f'\n{divider}')
print('  GROUP 7 — DEEPFAKE DETECTION SYSTEM — FINAL SUMMARY')
print(divider)
print(f'  Dataset mode : {"DEMO (synthetic data)" if DEMO_MODE else "REAL"}')
print(f'  Device       : {DEVICE}')
print()
print('  NFR COMPLIANCE:')
print(f'  NFR1 latency  ≤ {CFG["latency_budget_ms"]} ms  │ P95 = {lat_p95:.0f} ms  '
      f'{"✅" if lat_p95 < CFG["latency_budget_ms"] else "⚠️ "}')
print(f'  NFR2 AUC      ≥ {CFG["auc_target"]}      │ fused AUC = {fused_auc:.4f}  '
      f'{"✅" if fused_auc >= CFG["auc_target"] else "⚠️  (real data needed)"}')
print(f'  NFR2 FPR      ≤ {CFG["fpr_target"]}      │ video FPR = {vid_metrics["fpr"]:.4f}  '
      f'{"✅" if vid_metrics["fpr"] <= CFG["fpr_target"] else "⚠️ "}')
print(f'  NFR5 fairness ≤ {CFG["fairness_target"]}      │ see fairness_analysis.png')
print(f'  NFR6 repro.         │ MLflow + SEED={SEED} + requirements.txt')
print()
print('  MODEL AUC:')
print(f'  Video  : {vid_metrics["auc"]:.4f}')
print(f'  Audio  : {aud_metrics["auc"]:.4f}')
print(f'  Fused  : {fused_auc:.4f}')
print()
print('  SAVED ARTEFACTS:')
for fname in [
    'best_video_model.pth', 'best_audio_model.pth',
    'video_model_evaluation.png', 'audio_model_evaluation.png',
    'training_summary.png', 'latency_benchmark.png',
    'robustness_analysis.png', 'fairness_analysis.png',
]:
    print(f'  {"✅" if os.path.exists(fname) else "—"}  {fname}')
print(divider)
if DEMO_MODE:
    print()
    print('  ⚠️  Running on synthetic data. Upload real dataset to Drive')
    print('     (see Cell 0.2) for submission-quality results.')
    print(divider)
